In [ ]:
# Configuration
DIMENSIONS = {
    'spatial': 1,  # 1 or 2
    'velocity': 1   # 1 or 2
}


In [ ]:
def generate_basis(a, b, num_knots, degree, symbol, dim):
    """Generate 1D or 2D B-spline basis."""
    if dim == 1:
        return generate_1d_basis(a, b, num_knots, degree, symbol)
    elif dim == 2:
        return generate_2d_tensor_basis(a, b, num_knots, degree, symbol)

def generate_1d_basis(a, b, num_knots, degree, symbol):
    """1D basis (existing code)."""
    clamped_knots = generate_knot_vector(a, b, num_knots, degree)
    return [bspline_basis(degree, clamped_knots, i, symbol) for i in range(len(clamped_knots)-degree-1)], clamped_knots

def generate_2d_tensor_basis(a, b, num_knots, degree, symbols):
    """2D tensor product basis."""
    basis_x, knots_x = generate_1d_basis(a[0], b[0], num_knots[0], degree[0], symbols[0])
    basis_y, knots_y = generate_1d_basis(a[1], b[1], num_knots[1], degree[1], symbols[1])
    return [sp.Mul(phi_x, phi_y) for phi_x in basis_x for phi_y in basis_y], (knots_x, knots_y)


In [ ]:
def compute_overlap_matrix(basis, knots, degree, symbol, dim):
    """Compute M1/M2 for 1D or 2D."""
    if dim == 1:
        return compute_1d_overlap(basis, knots, degree, symbol)
    elif dim == 2:
        return compute_2d_kronecker_overlap(basis, knots, degree, symbol)

def compute_1d_overlap(basis, knots, degree, symbol):
    """Existing 1D code."""
    # ... (same as your original compute_M1_M2)

def compute_2d_kronecker_overlap(basis, knots, degree, symbol):
    """2D Kronecker product of 1D matrices."""
    M_x = compute_1d_overlap(basis[0], knots[0], degree[0], symbol[0])
    M_y = compute_1d_overlap(basis[1], knots[1], degree[1], symbol[1])
    return np.kron(M_x, M_y)


In [ ]:
def sample_positions(number_samples, dim):
    """Sample positions for 1D or 2D."""
    if dim == 1:
        return rejection_sampling_positions_1d(number_samples)
    elif dim == 2:
        return rejection_sampling_positions_2d(number_samples)

def sample_velocities(positions, dim):
    """Sample velocities for 1D or 2D."""
    if dim == 1:
        return [np.random.normal(u(x), np.sqrt(T(x))) for x in positions]
    elif dim == 2:
        return [np.random.normal(u(x, y), np.sqrt(T(x, y)), size=2) for x, y in positions]


In [ ]:
def compute_C_hat(phi_basis, psi_basis, x_samples, v_samples, dim):
    """Handle 1D1V or 2D2V data."""
    num_phi = len(phi_basis)
    num_psi = len(psi_basis)
    C_hat = np.zeros((num_phi, num_psi))
    
    # Lambdify with variable awareness
    if dim['spatial'] == 1:
        phi_funcs = [sp.lambdify(x, phi, 'numpy') for phi in phi_basis]
    else:
        phi_funcs = [sp.lambdify((x, y), phi, 'numpy') for phi in phi_basis]
        
    if dim['velocity'] == 1:
        psi_funcs = [sp.lambdify(v, psi, 'numpy') for psi in psi_basis]
    else:
        psi_funcs = [sp.lambdify((vx, vy), psi, 'numpy') for psi in psi_basis]
    
    for sample in zip(x_samples, v_samples):
        phi_vals = [f(*sample[0]) for f in phi_funcs]  # Unpack spatial dimensions
        psi_vals = [f(*sample[1]) for f in psi_funcs]  # Unpack velocity dimensions
        C_hat += np.outer(phi_vals, psi_vals)
    
    return C_hat / len(x_samples)


In [ ]:
# Configure dimensions
DIMENSIONS = {'spatial': 1, 'velocity': 1}  # Change to 2 for 2D2V

# Generate bases
if DIMENSIONS['spatial'] == 1:
    phi_basis, knots_x = generate_basis(x_a, x_b, num_knots_x, degree_x, x, dim=1)
else:
    phi_basis, knots_x = generate_basis((x_a, y_a), (x_b, y_b), (num_knots_x, num_knots_y), (degree_x, degree_y), (x, y), dim=2)

# Compute matrices
M1 = compute_overlap_matrix(phi_basis, knots_x, degree_x, x, dim=DIMENSIONS['spatial'])

# Sample data
x_samples = sample_positions(number_samples, dim=DIMENSIONS['spatial'])
v_samples = sample_velocities(x_samples, dim=DIMENSIONS['velocity'])

# Compute C_hat and C
C_hat = compute_C_hat(phi_basis, psi_basis, x_samples, v_samples, dim=DIMENSIONS)
C = np.linalg.inv(M1).T @ C_hat @ np.linalg.inv(M2)


In [3]:
a = [(1,2), (3,4)]
print(a[1][1])


4


In [4]:
x_a =1
x_b = 2
y_a = 3
y_b = 4
a =[(x_a,x_b),(y_a,y_b)]
print(a)

[(1, 2), (3, 4)]


In [6]:
print(a[1][1])


4
